# Phase 1 - XGBoost modeling for all five CO2 pressures

Single modeling notebook. Predicts CO2 uptake (mol/kg) at 0.01, 0.05, 0.1, 0.5 and 2.5 bar from four geometric features (`lcd`, `pld`, `void_fraction`, `surface_area_m2g`).

- Data: `data/processed/data_clean_v2`
- Model: XGBoost (one model per pressure), same parameters as `mof_models.XGBOOST_PARAMS`
- Split: 60/20/20 train/valid/test, seed 12345
- Outputs: `results/modeling_results/xgboost_all_pressures_metrics.csv`, models in `results/models/`

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Find the project root (works from the parent project and from the shared repo)
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "processed" / "data_clean_v2").exists()
)
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_v2"
RESULTS_DIR = PROJECT_ROOT / "results" / "modeling_results"
MODELS_DIR = PROJECT_ROOT / "results" / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

In [ ]:
FEATURES = ["lcd", "pld", "void_fraction", "surface_area_m2g"]
TARGETS = [
    "CO2_uptake_0.01bar_molkg",
    "CO2_uptake_0.05bar_molkg",
    "CO2_uptake_0.1bar_molkg",
    "CO2_uptake_0.5bar_molkg",
    "CO2_uptake_2.5bar_molkg",
]
SEED = 12345
XGB_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.1,
    "max_depth": 3,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "random_state": SEED,
    "n_jobs": 1,
}

df = pd.read_csv(DATA_PATH)
print(df.shape)
df[FEATURES + TARGETS].describe().T

In [ ]:
def split_data(frame, target):
    X, y = frame[FEATURES], frame[target]
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.4, shuffle=True, random_state=SEED
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, shuffle=True, random_state=SEED
    )
    return X_train, X_valid, X_test, y_train, y_valid, y_test


def metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": r2_score(y_true, y_pred),
    }

In [ ]:
rows, models, test_predictions = [], {}, {}

for target in TARGETS:
    X_train, X_valid, X_test, y_train, y_valid, y_test = split_data(df, target)
    model = XGBRegressor(**XGB_PARAMS).fit(X_train, y_train)

    row = {"target": target}
    for name, X, y in [("train", X_train, y_train), ("valid", X_valid, y_valid), ("test", X_test, y_test)]:
        for metric, value in metrics(y, model.predict(X)).items():
            row[f"{name}_{metric}"] = value
    rows.append(row)
    models[target] = model
    test_predictions[target] = (y_test, model.predict(X_test))

results = pd.DataFrame(rows)
results

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 4))
for ax, target in zip(axes, TARGETS):
    y_true, y_pred = test_predictions[target]
    ax.scatter(y_true, y_pred, s=4, alpha=0.4)
    lim = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lim, lim, "r--")
    ax.set_title(target.replace("CO2_uptake_", "").replace("_molkg", ""))
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
results.to_csv(RESULTS_DIR / "xgboost_all_pressures_metrics.csv", index=False)
for target, model in models.items():
    joblib.dump(model, MODELS_DIR / f"phase1_xgboost_{target}.joblib")
print("Saved metrics and", len(models), "models")